# Introduction to Renderers

A renderer defines the rules and symbols used to display the data in a layer. Renderers are used to create specific types of layer visualizations in a map or scene. For feature data, a single symbol can be defined for all features or multiple symbols can be defined based on different attribute values. Applying multiple symbols based on data is also called data-driven visualization. 

> Note: The rendering classes in the ArcGIS API for Python are based off the corresponding rendering classes in the ArcGIS Maps SDK for JavaScript and share a common data model. This change has been implemented in the interest of uniformity and to offer Python developers consistency in visual style that is available to JavaScript developers within the ArcGIS ecosystem. Where applicable, corresponding JavaScript implementations of these rendering classes will also be linked for reference.

In [2]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer

gis = GIS(profile="my_online_admin_profile")

Below is a list of some of the key rendering styles supported in the ArcGIS API for Python along with examples:

### 1.__`SimpleRenderer`__:
The [SimpleRenderer](https://developers.arcgis.com/python/latest/api-reference/arcgis.map.toc.html#simplerenderer) renders all features in a Layer with one Symbol. This renderer may be used to simply visualize the location of geographic features. For example, layers representing political boundaries, cities, buildings, rivers, etc. that are typically used for providing geographic context in an application may be defined with a single symbol per layer.

[SimpleRenderer in ArcGIS Maps SDK for JavaScript](https://developers.arcgis.com/javascript/latest/references/core/renderers/SimpleRenderer/)

We begin with a simple example showing state boundaries for the US.

In [91]:
states = FeatureLayer("https://services.arcgis.com/P3ePLMYs2RVChkJx/arcgis/rest/services/USA_States_Generalized_Boundaries/FeatureServer/0")

map1 = gis.map("United States")
map1.content.add(states)
map1

Here we will import the `SimpleRenderer` class and create a solid color renderer. This means each symbol will be defined by the same orange color with a white outline and custom width of the outline.

We then access the `RendererManager` for the layer we are working with by calling the `renderer_manager` property and passing the index of the layer on the map.

To set the renderer we assign our newly created `SimpleRenderer` called `simple_renderer` to the `renderer` property for the layer on the map.

> Note: For this and the following examples, setting this `renderer` property does not change the original layer's renderer, only the one represented in the map. 

In [88]:
from arcgis.map.renderers import SimpleRenderer

simple_renderer = SimpleRenderer(
    label="State boundaries",
    symbol={
        "type": "esriSFS",
        "style": "esriSFSSolid",
        "color": [255, 165, 0],
        "outline": {
            "type": "esriSLS",
            "style": "esriSLSSolid",
            "color": [255, 191, 0],
            "width": 0.75,
        },
    },
)

renderer_manager = map1.content.renderer_manager(0)
renderer_manager.renderer = simple_renderer

### 2. __`ClassBreaksRenderer`__:
[ClassBreaksRenderer](https://developers.arcgis.com/python/latest/api-reference/arcgis.map.toc.html#classbreaksrenderer) defines the symbol of each feature in a Layer based on the value of a numeric attribute. Symbols are assigned based on classes or ranges of data. Each feature is assigned a symbol based on the class break in which the value of the attribute falls.

[ClassBreaksRenderer in ArcGIS Maps SDK for JavaScript](https://developers.arcgis.com/javascript/latest/references/core/renderers/ClassBreaksRenderer/)

We will work with Starbucks data across Seattle here. While we create the Map object and add data to it, we also set the basemap value to `arcgis-dark-gray` which adds contrast to the renderer style we will set below.



In [98]:
starbucks = FeatureLayer("https://services.arcgis.com/q3Zg9ERurv23iysr/arcgis/rest/services/Starbucks_Location/FeatureServer/0")

map2 = gis.map("Seattle")
map2.content.add(starbucks)
map2.basemap.basemap = 'arcgis-dark-gray'
map2

For this example, we are importing the `ClassBreaksRenderer` and `ClassBreaksInfo` classes and we create custom `classes` based on the range of `Distance` values we would like to visualize distinctly, by providing different shades of green from light to dark to represent increasing ranges of distance.

We then create a `ClassBreaksRenderer` object and specify the `field` we have identified classes for, the minimum value or starting value, the default symbol which is a circle of style `esriSMSCircle` for data that is missing or incorrect. Note the `class_break_infos` parameter that not only defines the `esriSMSCircle` symbol for each data point (Starbucks location) but also provides the upper boundary, a legend label, and a symbol for each of the five classes we have previously defined.

We then access the `RendererManager` for the layer we are working with by calling the `renderer_manager` property and passing the index of the layer on the map.

To set the renderer we assign our newly created `ClassBreaksRenderer` called `class_breaks_renderer` to the `renderer` property for the layer on the map.

In [97]:
from arcgis.map.renderers import ClassBreaksRenderer, ClassBreakInfo

classes = [
    (10, "0–10", [237, 248, 233, 255]),
    (25, ">10–25", [186, 228, 179, 255]),
    (50, ">25–50", [116, 196, 118, 255]),
    (100, ">50–100", [49, 163, 84, 255]),
    (350, ">100–350", [0, 109, 44, 255]),
]

class_breaks_renderer = ClassBreaksRenderer(
    field="Distance",
    min_value=0,
    class_break_infos=[
        ClassBreakInfo(
            class_max_value=upper_bound,
            label=label,
            symbol={
                "type": "esriSMS",
                "style": "esriSMSCircle",
                "size": 9,
                "color": color,
                "outline": {
                    "type": "esriSLS",
                    "style": "esriSLSSolid",
                    "color": [80, 80, 80, 255],
                    "width": 0.5,
                },
            },
        )
        for upper_bound, label, color in classes
    ],
    default_symbol={
        "type": "esriSMS",
        "style": "esriSMSCircle",
        "size": 6,
        "color": [160, 160, 160, 255],
    },
)

renderer_manager = map2.content.renderer_manager(0)
renderer_manager.renderer = class_breaks_renderer

### 3. __`DotDensityRenderer`__:
[DotDensityRenderer](https://developers.arcgis.com/python/latest/api-reference/arcgis.map.toc.html#dotdensityrenderer) allows you to create dot density visualizations for polygon layers. Dot density visualizations randomly draw dots within each polygon to visualize the density of a population or some other variable. Each dot represents an approximate numeric value of an attribute or a subset of attributes.

[DotDensityRenderer in ArcGIS Maps SDK for JavaScript](https://developers.arcgis.com/javascript/latest/references/core/renderers/DotDensityRenderer/)

We will work with USA Census data for this renderer type.

In [104]:
item = gis.content.get("14c5450526a8430298b2fa74da12c2f4")
counties = item.layers[0]

map3 = gis.map("New York City")
map3.content.add(counties)
map3


This code creates a dot density renderer to visualize population across the New York City. Instead of assigning each area a single color based on its population, the renderer fills each area with dots, where each dot represents a specified number of people. This helps show the spatial distribution and concentration of population across the map.

We start by importing the `DotDensityRenderer` and `AttributeColorInfo` classes and creating `dot_density_renderer` by defining the population attributes via `AttributeColorInfo` by providing the attribution field, label and the shade of red we see on the map. We then provide the `dot_value` or how many people each dot represents, its size, `background_color` and `outline` style.

We then access the `RendererManager` for the layer we are working with by calling the `renderer_manager` property and passing the index of the layer on the map.


To set the renderer we assign our newly created `DotDensityRenderer` called `dot_density_renderer` to the `renderer` property for the layer on the map.

In [103]:
from arcgis.map.renderers import DotDensityRenderer, AttributeColorInfo

dot_density_renderer = DotDensityRenderer(
    attributes=[
        AttributeColorInfo(
            field="POPULATION",
            label="2020 population",
            color=[220, 30, 30, 255],
        )
    ],
    dot_value=1000,
    dot_size=1.5,
    background_color=[255, 255, 255, 0],
    outline={
        "type": "esriSLS",
        "style": "esriSLSSolid",
        "color": [150, 150, 150, 150],
        "width": 0.4,
    }
)

renderer_manager = map3.content.renderer_manager(0)
renderer_manager.renderer = dot_density_renderer

### 4. __`HeatmapRenderer`__: 
The [HeatmapRenderer](https://developers.arcgis.com/python/latest/api-reference/arcgis.map.toc.html#heatmaprenderer) renders point data into a raster visualization that emphasizes areas of higher density or weighted values. It uses kernel density to render point features and density at each pixel is calculated by adding the values of all the kernel surfaces where they overlay.

[HeatmapRenderer in ArcGIS Maps SDK for JavaScript](https://developers.arcgis.com/javascript/latest/references/core/renderers/HeatmapRenderer/)

We will look at data for alternative fuel stations across the US.

In [13]:
item = gis.content.get("4a8305b53ed7416795fc6a129ec989ee")
stations = item.layers[0]

map4 = gis.map("California")
map4.content.add(stations)
map4

In [9]:
from arcgis.map.renderers import HeatmapRenderer, HeatmapColorStop

heatmap_renderer = HeatmapRenderer(
    color_stops=[
        HeatmapColorStop(ratio=0.00, color=[145, 150, 170, 150]),
        HeatmapColorStop(ratio=0.15, color=[165, 140, 155, 160]),
        HeatmapColorStop(ratio=0.30, color=[195, 95, 160, 180]),
        HeatmapColorStop(ratio=0.45, color=[225, 35, 140, 210]),
        HeatmapColorStop(ratio=0.60, color=[240, 0, 60, 240]),
        HeatmapColorStop(ratio=0.75, color=[255, 40, 0, 255]),
        HeatmapColorStop(ratio=0.90, color=[255, 190, 0, 255]),
        HeatmapColorStop(ratio=1.00, color=[255, 255, 0, 255]),
    ],
    radius=20,
    min_density=0,
    max_density=0.01
)

renderer_manager = map4.content.renderer_manager(0)
renderer_manager.renderer = heatmap_renderer

We import `HeatmapRenderer` and `HeatmapColorStop` classes and proceed to build our renderer object. `HeatmapColorStop` defines a color at a particular density level. The `color_stops` list creates a gradient that transitions from muted gray-purple through pink and red to orange and yellow. The `ratio` value indicates the relative position of each color stop along the heatmap's density scale, ranginf from 0.00 to 1.00. We also set `radius` that determines how far beyond itself a point could have influence. A lower value for `radius` causes points to cluster together and create a dense heatmap. We then set minimum and maximum values for density.

We then access the `RendererManager` for the layer we are working with by calling the `renderer_manager` property and passing the index of the layer on the map.

To set the renderer we assign our newly created `HeatmapRenderer` called `heatmap_renderer` to the `renderer` property for the layer on the map.

### 5. __`PieChartRenderer`__:
The [PieChartRenderer](https://developers.arcgis.com/python/latest/api-reference/arcgis.map.toc.html#piechartrenderer) allows you to create a pie chart for each feature in the layer. The value and color of each pie slice is specified in the attributes property.

[PieChartRenderer in ArcGIS Maps SDK for JavaScript](https://developers.arcgis.com/javascript/latest/references/core/renderers/PieChartRenderer/)

We will build a map with this renderer type using data for renters v/s owners for each state across the US. While we create the Map object and add data to it, we also set the basemap value to `arcgis-dark-gray` which adds contrast to the renderer style we will set below.

In [12]:
states = FeatureLayer("https://sampleserver6.arcgisonline.com/arcgis/rest/services/Census/MapServer/3")

map5 = gis.map("United States")
map5.content.add(states)
map5.basemap.basemap = 'arcgis-dark-gray'
map5

This code creates a `PieChartRenderer` object to visualize the composition of housing occupancy across the US. Each feature is represented by a pie chart showing the relative amounts of owner-occupied and renter-occupied housing for each state.

We use `AttributeColorInfo` again to set owner occupied housing attributes to yellow color and renter occupied housing to blue color. We then set the size of each pie, its white outline, `background_fill_symbol` to contrast the pies against and a default color for missing data.

We then access the `RendererManager` for the layer we are working with by calling the `renderer_manager` property and passing the index of the layer on the map.

To set the renderer we assign our newly created `PieChartRenderer` called `pie_chart_renderer` to the `renderer` property for the layer on the map.

In [11]:
from arcgis.map.renderers import PieChartRenderer, AttributeColorInfo

pie_chart_renderer = PieChartRenderer(
    attributes=[
        AttributeColorInfo(
            field="OWNER_OCC",
            label="Owner-occupied housing",
            color=[255, 215, 0, 255],
        ),
        AttributeColorInfo(
            field="RENTER_OCC",
            label="Renter-occupied housing",
            color=[49, 130, 189, 255],
        ),
    ],
    size=48,
    hole_percentage=0,
    outline={
        "type": "esriSLS",
        "style": "esriSLSSolid",
        "color": [255, 255, 255, 255],
        "width": 1,
    },
    background_fill_symbol={
        "type": "esriSFS",
        "style": "esriSFSSolid",
        "color": [235, 235, 235, 100],
        "outline": {
            "type": "esriSLS",
            "style": "esriSLSSolid",
            "color": [130, 130, 130, 255],
            "width": 0.6,
        },
    },
    default_color=[160, 160, 160, 255]
)


renderer_manager = map5.content.renderer_manager(0)
renderer_manager.renderer = pie_chart_renderer

### 6. __`UniqueValueRenderer`__:
The [UniqueValueRenderer](https://developers.arcgis.com/python/latest/api-reference/arcgis.map.toc.html#uniquevaluerenderer)  allows you to symbolize features in a Layer based on one or more categorical attributes. This is typically done by using unique colors, fill styles, or images to represent features with equal values in a string field.

[UniqueValueRenderer in ArcGIS Maps SDK for JavaScript](https://developers.arcgis.com/javascript/latest/references/core/renderers/UniqueValueRenderer/)

For the sake of simplicity, we will filter the Alternative fuel stations data for this example to focus only on stations in the state of California.

In [40]:
item = gis.content.get("4a8305b53ed7416795fc6a129ec989ee")
stations = item.layers[0]

stations_ca = stations.query(
    where=(
        "State = 'CA' AND "
        "(EV_Level2_EVSE_Ports > 0 OR EV_DC_Fast_Ports > 0)"
    ),
    out_fields="City,Station_Name,EV_Level2_EVSE_Ports,EV_DC_Fast_Ports",
    result_record_count=8,
)

map6 = gis.map("California")
map6.content.add(stations_ca)
map6.zoom_to_layer(stations_ca)
map6

This code creates a unique value renderer to display 8 alternative fuel stations in California (stations_ca) using different colors based on their `City` attribute. Each city is assigned a distinct color, allowing stations from different cities to be visually distinguished on the map.

We import the `UniqueValueRenderer` and `UniqueValueInfo` classes and start by creating a circular symbol, of size 18 with dark grey color within a `marker()` function. The `cities` variable stores distinct city names in an alphabetically sorted list. Since we know that we have filtered just 8 data points, we can set a color palette for 8 distinct colors in the `list` called `colors`. We then use the city name in the `value_expression` parameter of the `unique_renderer` object we create to assign color and style based on the city and the corresponding color for that city set vis `zip(cities, color)` and styled per the set values in the `marker()` function.

We then access the `RendererManager` for the layer we are working with by calling the `renderer_manager` property and passing the index of the layer on the map.

To set the renderer we assign our newly created `UniqueValueRenderer` called `unique_renderer` to the `renderer` property for the layer on the map.

In [37]:
from arcgis.map.renderers import UniqueValueRenderer, UniqueValueInfo

def marker(color):
    return {
        "type": "esriSMS",
        "style": "esriSMSCircle",
        "size": 18,
        "color": color,
        "outline": {
            "type": "esriSLS",
            "style": "esriSLSSolid",
            "color": [60, 60, 60, 255],
            "width": 1,
        },
    }

cities = sorted({
    feature.attributes["City"]
    for feature in stations_ca.features
    if feature.attributes["City"] is not None
})

colors = [
    [255, 215, 0, 255],
    [49, 130, 189, 255],
    [49, 163, 84, 255],
    [230, 85, 13, 255],
    [117, 107, 177, 255],
    [220, 80, 150, 255],
    [0, 180, 190, 255],
    [130, 100, 60, 255],
]

unique_renderer = UniqueValueRenderer(
    value_expression='$feature["City"]',
    unique_value_infos=[
        UniqueValueInfo(
            value=city,
            label=city,
            symbol=marker(color),
        )
        for city, color in zip(cities, colors)
    ],
    default_symbol=marker([160, 160, 160, 255])
)

renderer_manager = map6.content.renderer_manager(0)
renderer_manager.renderer = unique_renderer

### 7. __`PredominanceRenderer`__:
The [PredominanceRenderer](https://developers.arcgis.com/python/latest/api-reference/arcgis.map.toc.html#predominancerenderer) is a type of UniqueValue renderer which is based off the `valueExpression` property rather than field. Optionally, size and/or transparency visual variables may be included with valueExpression. Note that this renderer is supported for ArcGIS Online hosted feature services and feature collections.

[Predominance in ArcGIS Maps SDK for JavaScript](https://developers.arcgis.com/javascript/latest/visualization/data-driven-styles/predominance/)

To understand the difference, let's work with the same data we used in the previous example for `UniqueValueRenderer`.

In [61]:
item = gis.content.get("4a8305b53ed7416795fc6a129ec989ee")
stations = item.layers[0]

stations_ca = stations.query(
    where=(
        "State = 'CA' AND "
        "(EV_Level2_EVSE_Ports > 0 OR EV_DC_Fast_Ports > 0)"
    ),
    out_fields="City,Station_Name,EV_Level2_EVSE_Ports,EV_DC_Fast_Ports",
    order_by_fields="City ASC",
    result_record_count=8,
)

map7 = gis.map("California")
map7.content.add(stations_ca)
map7.zoom_to_layer(stations_ca)
map7

This code creates a `PredominanceRenderer` object to identify which type of electric vehicle (EV) charging port is more common at each station. It compares the number of Level 2 charging ports with the number of DC fast-charging ports and assigns each station a color based on which type has the higher count. Unlike the previous unique value renderer, which groups stations by city, this renderer classifies stations according to the relationship between two numeric attributes.

The `value_expression` written in Arcade gives us one of the three values `Level2`, `Fast` or `Tie` based on dominance of charger type.

We will assign colors to each of those 3 values to signify the following:
* Yellow: more Level 2 ports at the station.
* Blue: more DC fast ports at the station.
* Gray: both are equally available at the station.

We then access the `RendererManager` for the layer we are working with by calling the `renderer_manager` property and passing the index of the layer on the map.

To set the `renderer` we assign our newly created `PredominanceRenderer` called `predominance_renderer` to the `renderer` property for the layer on the map.

In [58]:
from arcgis.map.renderers import PredominanceRenderer

predominance_renderer = PredominanceRenderer(
    value_expression="""
        var level2 = DefaultValue($feature.EV_Level2_EVSE_Ports, 0);
        var fast = DefaultValue($feature.EV_DC_Fast_Ports, 0);

        return When(
            level2 > fast, "Level2",
            fast > level2, "Fast",
            "Tie"
        );
    """,
    value_expression_title="Predominant charging-port type",
    unique_value_infos=[
        UniqueValueInfo(
            value="Level2",
            label="More Level 2 ports",
            symbol=marker([255, 215, 0, 255]),
        ),
        UniqueValueInfo(
            value="Fast",
            label="More DC fast ports",
            symbol=marker([49, 130, 189, 255]),
        ),
        UniqueValueInfo(
            value="Tie",
            label="Equal counts",
            symbol=marker([160, 160, 160, 255]),
        ),
    ],
)

renderer_manager = map7.content.renderer_manager(0)
renderer_manager.renderer = predominance_renderer

### Conclusion

To learn more about the various classes related to rendering in the ArcGIS API for Python, please refer to the [API reference documentation for `Renderers`](https://developers.arcgis.com/python/latest/api-reference/arcgis.map.toc.html#renderers).

Below are a few related resources to further your understanding and familiarity with this concept:
1. [Advanced Cartography 1](https://developers.arcgis.com/python/latest/guide/advanced-cartography-part1/)
2. [Advanced Cartography 2](https://developers.arcgis.com/python/latest/guide/advanced-cartography-part2/)
3. [Smart mapping](https://developers.arcgis.com/python/latest/guide/smart-mapping/)